## To-Do:
1. Refine Match Extraction for Speicifc Tournaments
- Need to Make sure Matches are extracted for appropraite torunaments (e.g. Regional Qualifiers that exist within World Cup Pages)
- Need to extract matches that don't conform to standard Date `Match - Results - Location` Format

In [2]:
import json
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
from copy import deepcopy

import re
from wikipedia_scraper import fetch_wikipedia_page, search_wikipedia

In [3]:
wru_data_json = json.load(open("data/cleaned_data/wru_yearly_breakdown_added_tournament_page_names.json"))

In [5]:
test_nations = set()
for year, details in wru_data_json.items():
    other_matches = details['other_matches']
    for i, match in other_matches.items():
        match['team_a'] = match['team_a'].replace('\xa0', '')
        match['team_b'] = match['team_b'].replace('\xa0', '')
        
        if 'Trinidad' in match['team_a']:
            match['team_a'] = 'Trinidad and Tobago'
        if 'Trinidad' in match['team_b']:
            match['team_b'] = 'Trinidad and Tobago'
        
        if 'Caribbean Select' in match['team_a']:
            match['team_a'] = 'Caribbean Select XV'
        if 'Caribbean Select' in match['team_b']:
            match['team_b'] = 'Caribbean Select XV'

        if match['team_a'] not in test_nations:
            test_nations.add(match['team_a'])
        if match['team_b'] not in test_nations:
            test_nations.add(match['team_b'])

In [14]:
for year, details in wru_data_json.items():
    tournaments = details['tournaments']
    if tournaments != None:
        for i, tournament in tournaments.items():
            print(f"Extracting Page Details for: {tournament['wikipedia_page_name']}")
            tournament['wikipedia_page_details'] = fetch_wikipedia_page(tournament['wikipedia_page_name'])

Extracting Page Details for: 1988 Women's Rugby European Cup
Extracting Page Details for: RugbyFest 1990
Extracting Page Details for: 1991 Women's Rugby World Cup
Extracting Page Details for: Canada Cup 1993
Extracting Page Details for: 1994 Women's Rugby World Cup
Extracting Page Details for: 1994 Laurie O'Reilly Cup
Extracting Page Details for: 1995 FIRA Women's European Championship
Extracting Page Details for: 1995 Laurie O'Reilly Cup
Extracting Page Details for: 1996 Women's Home Nations Championship
Extracting Page Details for: 1996 FIRA Women's European Championship
Extracting Page Details for: 1996 Laurie O'Reilly Cup
Extracting Page Details for: 1996 Canada Cup
Extracting Page Details for: 1997 Women's Home Nations Championship
Extracting Page Details for: 1997 FIRA Women's European Championship
Extracting Page Details for: 1997 Laurie O'Reilly Cup
Extracting Page Details for: 1998 Women's Home Nations Championship
Extracting Page Details for: 1998 Women's Rugby World Cup
Extr

KeyError: 'parse'

In [32]:
for year, details in wru_data_json.items():
    tournaments = details['tournaments']
    if tournaments != None:
        for i, tournament in tournaments.items():
            print(year)
            print(tournament['tournament_name'])
            print(tournament['wikipedia_page_name'])
            tournament['wikipedia_page_details'] = tournament['wikipedia_page_details'].replace('\xa0', '')
            tournament_details = tournament['wikipedia_page_details']
            #print(tournament['wikipedia_page_details'])
            tournament_matches = re.findall(r"\d{4}.\d{2}.\d{1,2}(?:[A-Z][a-z]*\s)?[A-Z][a-z]*.?\d*.\d*\s?(?:[A-Z][a-z]*\?)?[A-Z][a-z]*.*|\d{1,}.[A-Z][a-z]*.\d{4}(?:[A-Z][a-z]*\s)?[A-Z][a-z]*.?\d*.\d*\s?(?:[A-Z][a-z]*\?)?[A-Z][a-z]*.*", tournament_details)
            print(tournament_matches)
            print('\n')

1988
European Cup
1988 Women's Rugby European Cup
['1988-05-21France13–3NetherlandsBourg en Bresse', '1988-05-21Great Britain32–9ItalyBourg en Bresse', '1988-05-22France16–3ItalyBourg en Bresse', '1988-05-22Great Britain26–0NetherlandsBourg en Bresse', '1988-05-23France8–6Great BritainBourg en Bresse', '1988-05-23Italy6–10NetherlandsBourg en Bresse']


1990
RugbyFest 1990
RugbyFest 1990
['26 August 1990New Zealand56–0NetherlandsChristchurch', '28 August 1990New Zealand8–0Soviet UnionChristchurch', '29 August 1990Netherlands0–38United StatesChristchurch', '30 August 1990Netherlands12–4Soviet UnionChristchurch', '30 August 1990New Zealand9–3United StatesChristchurch', '31 August 1990United States32–0Soviet UnionChristchurch', '1 September 1990New Zealand12–4 World XVChristchurch', '19 August 1990Canterbury "B"4-12Netherlands', '20 August 1990Crusadettes20-0Netherlands', '20 August 1990Burnside-Merrivale6-12Netherlands', '21 August 1990Canterbury "A"34-4Netherlands', '22 August 1990Nether

KeyError: 'wikipedia_page_details'

In [ ]:
"Canada Cup 1993"
"1994 Laurie O'Reilly Cup"
"1995 Laurie O'Reilly Cup"
"1996 Women's Home Nations Championship"
"1996 Laurie O'Reilly Cup"
"1997 Women's Home Nations Championship"
"1997 Laurie O'Reilly Cup"
"1998 Women's Home Nations Championship"
"1998 Laurie O'Reilly Cup"
"1999 Women's Home Nations Championship"
"Triangular '99"
"2000 Women's Home Nations Championship"
"2001 Women's Home Nations Championship"
"2000 Asian World Cup qualifier" = "2002 Women's Home Nations Championship"

In [ ]:
for year in range(1982, current_year+1):
    year_dict = {}
    
    year_str = str(year)
    match = re.findall(rf"({year_str}\[edit\].*?){year+1}\[edit\]|({year_str}\[edit\].*?)Notes\[edit\]", page_details, flags=re.DOTALL|re.I)

    if match[0][0] != '':
        details = match[0][0]
    else:
        details = match[0][1]

    tournament_match = re.findall(r"Tournaments\[edit\](.*)Other", details, flags=re.DOTALL|re.I)[0]
    if tournament_match == "\nNone\n" or tournament_match == "\nNone\n\n":
        year_dict['tournaments'] = None
    else:
        year_dict['tournaments'] = tournament_match
    
    other_matches_match = re.findall(r"Other matches\[edit\](.*)", details, flags=re.DOTALL|re.I)[0]
    year_dict['other_matches'] = other_matches_match
    
    
    full_dict[year_str] = year_dict

In [ ]:
for year, details in wru_yearly_breakdown_json.items():
    tournaments_dict = {}
    tournaments = details['tournaments']

    other_matches_dict = {}
    other_matches = details['other_matches']
    for i, test in enumerate(re.findall(r"(.*)", other_matches)):
        test = test.replace('[a]', '')
        test = test.replace('[b]', '')
        test = test.replace(' [1]', '')
        test = test.replace(' [2]', '')
        test = test.replace(' [3]', '')
        if test != ''and 'main article' not in test.lower():
            match_dict = {}
            try:
                match_dict['match_date'] = re.findall(r"(\d{1,2}\s[a-zA-Z]*\s\d{4})", test)[0]
            except:
                match_dict['match_date'] = re.findall(r"(\?{1,2}\s\?{1,2}\s\d{4})", test)[0]
            try:
                match_dict['team_a'] = re.findall(fr"(?<={other_matches_dict['match_date']})(?:\[N 1\])?\s*?((?:[A-Z][a-z]*\s)?[a-zA-Z]*)", test)[0]
            except:
                match_dict['team_a'] = re.findall(r"(?<=\d{4})(?:\[N 1\])?\s*?((?:[A-Z][a-z]*\s)?[a-zA-Z]*)", test)[0]
                
            match_dict['score'] = re.findall(r"(\d{1,}–\d{1,})", test)[0]
            team_b =  re.findall(fr"(?<={match_dict['score']}\s).*?((?:[A-Z][a-z]*\s)?[A-Z][a-z]*)", test)[0]
            
            if team_b == 'World X':
                match_dict['team_b'] = 'World XV'
            else:
                match_dict['team_b'] = team_b

            if re.findall(r",\s?(.*)", test):
                match_dict['location'] = re.findall(r",\s?(\w*)[A-Z]?", test)[0]
            else:
                match_dict['location'] = re.findall(fr"(?<={match_dict['team_b']})\s?(.*)", test)[0]
            other_matches_dict[i] = match_dict
    
    cleaned_dict[year] = {'tournaments': tournaments_dict, 'other_matches' : other_matches_dict}